# 0. Split Edges

This notebook preprocesses the raw transaction edge data (`edge_rek_credit` and `edge_rek_debit`).
It splits them into `simpanan` and `pinjaman` specific edges based on account number patterns.
This is a prerequisite for subsequent indexing steps.

In [ ]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
RAW_DATA_DIR = os.path.join(ROOT_DIR, "data")

# Input Directories
EDGE_REK_CREDIT_DIR = os.path.join(RAW_DATA_DIR, "edge_rek_credit")
EDGE_REK_DEBIT_DIR = os.path.join(RAW_DATA_DIR, "edge_rek_debit")

# Output Directories (Will be created if not exist in RAW_DATA_DIR)
OUTPUT_SIMP_CREDIT_DIR = os.path.join(RAW_DATA_DIR, "edge_simp_credit")
OUTPUT_PINJ_CREDIT_DIR = os.path.join(RAW_DATA_DIR, "edge_pinj_credit")
OUTPUT_SIMP_DEBIT_DIR = os.path.join(RAW_DATA_DIR, "edge_simp_debit")
OUTPUT_PINJ_DEBIT_DIR = os.path.join(RAW_DATA_DIR, "edge_pinj_debit")

# Ensure output directories exist
os.makedirs(OUTPUT_SIMP_CREDIT_DIR, exist_ok=True)
os.makedirs(OUTPUT_PINJ_CREDIT_DIR, exist_ok=True)
os.makedirs(OUTPUT_SIMP_DEBIT_DIR, exist_ok=True)
os.makedirs(OUTPUT_PINJ_DEBIT_DIR, exist_ok=True)

In [ ]:
# Imports
import os
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
# Utility Function
def parsing_norek(nomor_rekening):
    """
    Deterministically identifies if an account is 'simpanan', 'pinjaman', or 'lainnya'
    based on the 3rd to last digit.
    """
    nomor_rekening = str(nomor_rekening)
    if len(nomor_rekening) < 3:
        return "lainnya"
        
    # Check 3rd character from end (1-based index 3 from right is -3)
    # E.g. ...5XX -> Simpanan, ...1XX -> Pinjaman
    digit = nomor_rekening[-3]
    
    if digit == "5":
        return "simpanan"
    elif digit == "1":
        return "pinjaman"
    else:
        return "lainnya"

In [ ]:
def process_edge_partitions(input_dir, output_simp_dir, output_pinj_dir, account_col="dst"):
    """
    Reads parquet partitions from input_dir, splits them, and writes to output dirs.
    account_col: which column contains the account number to check ('src' or 'dst')
    """
    if not os.path.exists(input_dir):
        print(f"Input directory not found: {input_dir}")
        return

    partitions = os.listdir(input_dir)
    partitions = [p for p in partitions if p.endswith('.parquet')]
    
    print(f"Processing {len(partitions)} partitions from {input_dir}...")

    for partition in tqdm(partitions):
        path = os.path.join(input_dir, partition)
        try:
            temp = pd.read_parquet(path)
            
            if account_col not in temp.columns:
                print(f"Warning: '{account_col}' not found in {partition}. Skipping.")
                continue

            temp["flag_rekening"] = temp[account_col].apply(parsing_norek)

            simp = temp[temp["flag_rekening"] == "simpanan"].drop(columns=["flag_rekening"])
            pinj = temp[temp["flag_rekening"] == "pinjaman"].drop(columns=["flag_rekening"])

            if not simp.empty:
                simp.to_parquet(os.path.join(output_simp_dir, partition), index=False)
            if not pinj.empty:
                pinj.to_parquet(os.path.join(output_pinj_dir, partition), index=False)
                
        except Exception as e:
            print(f"Error processing {partition}: {e}")

In [ ]:
# Process Credit Edges (Account is in DST)
print("Processing Credit Edges (Split by DST account)...")
process_edge_partitions(
    EDGE_REK_CREDIT_DIR, 
    OUTPUT_SIMP_CREDIT_DIR, 
    OUTPUT_PINJ_CREDIT_DIR, 
    account_col="dst"
)

In [ ]:
# Process Debit Edges (Account is in SRC)
print("Processing Debit Edges (Split by SRC account)...")
process_edge_partitions(
    EDGE_REK_DEBIT_DIR, 
    OUTPUT_SIMP_DEBIT_DIR, 
    OUTPUT_PINJ_DEBIT_DIR, 
    account_col="src"
)

In [ ]:
print("Done.")